# Cahen-Fourot et al. (2020) replication

Notebook scaffold to recreate the inverted pyramid analysis using monetary MRIO tables. Core computations live in `mrio_pyramids` so this notebook stays lean. Fill in data paths and run cells once datasets are available.

## Setup
- Install dependencies (once): `pip install -r requirements.txt`
- Ensure the MRIO dataset is downloaded (EXIOBASE3 or WIOD). The helper in `mrio_pyramids.data.load_exiobase3` can download EXIOBASE3 if network access is allowed.

In [ ]:
import sys
from pathlib import Path

ROOT = Path().resolve()
if (ROOT / 'src').exists():
    sys.path.append(str(ROOT / 'src'))
elif (ROOT.parent / 'src').exists():
    sys.path.append(str(ROOT.parent / 'src'))

import pandas as pd
import pymrio
import mrio_pyramids as mp
from mrio_pyramids import data

pd.set_option("display.max_rows", 20)
pd.set_option("display.max_columns", 20)

## Load MRIO system
If running without network, point `mrio_path` to a locally stored EXIOBASE3/WIOD parsed directory.

In [ ]:
from pathlib import Path
import pymrio

# dataset options: "oecd" (default), "test" (tiny demo), "exiobase" (public), "wiod" (requires IO file)
DATASET = "oecd"
YEAR = 2018
REGION = "DEU"  # set to None to use full system; for OECD, use country codes (e.g., DEU, FRA, GBR, ITA)

if DATASET == "wiod":
    data_dir = Path(f"data/wiod13_{YEAR}")
    pkl_path = data_dir / "mrio_system.pkl"
elif DATASET == "exiobase":
    data_dir = Path(f"data/exiobase3_{YEAR}")
    pkl_path = data_dir / "mrio_system.pkl"
elif DATASET == "oecd":
    data_dir = Path(f"data/oecd_icio_{YEAR}")
    pkl_path = data_dir / f"mrio_system_{YEAR}.pkl"
elif DATASET == "test":
    data_dir = Path("data/test")
    pkl_path = None
else:
    raise ValueError("Unsupported DATASET")

mrio = None
if pkl_path and pkl_path.exists():
    print(f"Loading cached MRIO from {pkl_path}")
    mrio = pymrio.read_mrio(pkl_path)
    try:
        mrio.calc_all()
    except Exception:
        pass
elif DATASET == "test":
    mrio = pymrio.load_test()
    mrio.calc_all()
else:
    data_dir.mkdir(parents=True, exist_ok=True)
    print(f"Downloading {DATASET.upper()} {YEAR} (may take minutes)...")
    if DATASET == "wiod":
        pymrio.download_wiod2013(storage_folder=data_dir, years=[YEAR])
        mrio = pymrio.parse_wiod(data_dir, year=YEAR)
    elif DATASET == "oecd":
        pymrio.download_oecd(storage_folder=data_dir, years=[YEAR])
        mrio = pymrio.parse_oecd(data_dir, year=YEAR)
    else:
        pymrio.download_exiobase3(storage_folder=data_dir, years=[YEAR], system="pxp")
        mrio = pymrio.parse_exiobase3(data_dir / f"IOT_{YEAR}_pxp.zip")
    mrio.calc_all()
    if pkl_path:
        mrio.save(pkl_path)
        print(f"Saved MRIO to {pkl_path}")


## Build Leontief/Ghosh and final demand
Select final demand categories to match the paper (e.g., household consumption, government, gross fixed capital formation, exports). Compute both Leontief (demand-driven) and Ghosh (supply-driven) inverses.

In [ ]:
if mrio is not None:
    # Match paper: exclude exports from final demand (adjust if needed)
    y = data.final_demand_vector(
        mrio,
        demand_categories=[c for c in mrio.Y.columns if "EXP" not in c],
    )

    L = mp.compute_leontief(mrio.Z, mrio.x)
    G = mp.compute_ghosh(mrio.Z, mrio.x)

    pyramid_inputs = mp.PyramidInputs(leontief=L, final_demand=y)

    print("Leontief shape:", L.shape)
    print("Ghosh shape:", G.shape)
    print("Final demand length:", len(y))


## Monetary inverted pyramid (Leontief vs Ghosh)
Compute upstream contributions supporting each sector's final demand (monetary base) using Leontief, and supply-side propagation using Ghosh.

In [ ]:
if mrio is not None:
    monetary_base = mp.embodied_pressure(pyramid_inputs)
    print("Top upstream contributors (Leontief demand-driven):")
    display(mp.rank_foundational_sectors(monetary_base, top_n=15))

    # Supply-driven propagation using Ghosh: apply final demand as injections
    ghosh_response = (G.values @ y.values)
    ghosh_base = mp.rank_foundational_sectors(
        pd.Series(ghosh_response, index=G.index, name="ghosh_base"), top_n=15
    )
    print("Top upstream contributors (Ghosh supply-driven):")
    display(ghosh_base)

    allocations = mp.dependency_contributions(pyramid_inputs, focus_sectors=mrio.Z.index[:10])
    allocations.head()


## Visual checks
Plug in your preferred plotting (bar, treemap, network). Keep plotting functions in `mrio_pyramids/plotting.py` if they grow.

## Dependency graph and clustering
Build a pruned dependency graph (supply-driven via Ghosh) and detect communities.

In [ ]:
if mrio is not None:
    # Build supply-driven dependency graph from Ghosh matrix
    ghosh_df = pd.DataFrame(G, index=mrio.Z.index, columns=mrio.Z.columns)
    dep_graph = mp.build_dependency_graph(
        ghosh_df,
        top_k_per_downstream=3,   # keep strongest suppliers per sector
        threshold=0.01,           # drop tiny links
        normalize_by_column=True,
    )
    undirected = mp.symmetrize(dep_graph, agg="max")
    communities = mp.detect_communities(undirected)
    print(f"Communities detected: {len(set(communities.values()))}")
